# CSC Machine Learning — Semester Project: Neural Networks
**Due Date: 5/25/26**

## Task: Language Modeling with Project Gutenberg

This project trains word-level neural language models on classic literature from Project Gutenberg.
A language model predicts the next word given a fixed-length context window of previous words.
I compare three neural architectures evaluated using **perplexity** — the standard metric for language models.

| Model | Architecture |
|---|---|
| Model 1 | Dense Neural Network (Embedding + Flatten + Dense) |
| Model 2 | LSTM — Long Short-Term Memory |
| Model 3 | GRU — Gated Recurrent Unit |

**Dataset:** 18 public-domain books from Project Gutenberg via NLTK (~2.3M tokens, auto-downloaded).

---
# Part 0: Environment Setup

In [ ]:
import nltk
nltk.download("gutenberg", quiet=True)

import numpy as np
import tensorflow as tf
from tensorflow import keras
from nltk.corpus import gutenberg
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)
print("GPU available     :", tf.config.list_physical_devices("GPU"))

---
# Part 1: Dataset Exploration

## 1.1 — Load Raw Data

The NLTK Gutenberg corpus contains 18 books from Project Gutenberg spanning fiction, non-fiction,
the Bible, and Shakespeare. Using all 18 books gives the model diverse vocabulary and sentence structures.

In [ ]:
book_ids = gutenberg.fileids()
print(f"Number of books: {len(book_ids)}")

book_stats = {}
all_raw_words = []
for bid in book_ids:
    words = list(gutenberg.words(bid))
    book_stats[bid] = len(words)
    all_raw_words.extend(words)

print(f"Total raw tokens : {len(all_raw_words):,}")
print(f"Unique raw tokens: {len(set(all_raw_words)):,}")
print()
print("Words per book:")
for bid, count in sorted(book_stats.items(), key=lambda x: -x[1]):
    print(f"  {bid:<35} {count:>8,}")

## 1.2 — Dataset Characteristics Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_books = sorted(book_stats.items(), key=lambda x: x[1])
short_names  = [bid.replace(".txt","").replace("-"," ").replace("_"," ") for bid, _ in sorted_books]
counts       = [c for _, c in sorted_books]

bars = axes[0].barh(short_names, counts, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Word Count")
axes[0].set_title("Words per Book — Project Gutenberg Corpus", fontweight="bold")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_width() + 3000, bar.get_y() + bar.get_height()/2,
                 f"{count:,}", va="center", fontsize=7)

alpha_count = sum(1 for w in all_raw_words if w.isalpha())
punct_count = len(all_raw_words) - alpha_count
axes[1].pie([alpha_count, punct_count],
            labels=["Alphabetic (kept)", "Punct/Other (removed)"],
            colors=["steelblue", "salmon"], explode=[0, 0.05],
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Token Composition Before Preprocessing", fontweight="bold")

plt.tight_layout()
plt.savefig("vis1_dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Alphabetic : {alpha_count:,}  ({alpha_count/len(all_raw_words):.1%})")
print(f"Punct/other: {punct_count:,} ({punct_count/len(all_raw_words):.1%})")

## 1.3 — Word Frequency Analysis (Zipf's Law)

Natural language follows **Zipf's Law**: word frequency drops sharply as rank increases.
On a log-log scale this is a straight line. This justifies our vocabulary cutoff:
the top 10,000 words cover the vast majority of the corpus.

In [ ]:
raw_freq  = Counter(all_raw_words)
all_freqs = sorted(raw_freq.values(), reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ranks = np.arange(1, len(all_freqs) + 1)
axes[0].loglog(ranks, all_freqs, color="steelblue", linewidth=1.5, alpha=0.8)
axes[0].axvline(10000, color="red", linestyle="--", linewidth=1.5, label="Vocab cutoff (10,000)")
axes[0].set_xlabel("Word Rank (log scale)")
axes[0].set_ylabel("Frequency (log scale)")
axes[0].set_title("Zipfs Law — Word Frequency Distribution", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

top30 = raw_freq.most_common(30)
axes[1].barh([w for w,_ in top30][::-1], [c for _,c in top30][::-1], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Frequency")
axes[1].set_title("Top 30 Most Frequent Tokens (raw corpus)", fontweight="bold")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("vis2_word_frequency.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Part 2: Data Preprocessing

## 2.1 — Preprocessing Steps

1. **Lowercase normalization** — `The` and `the` are the same word; treating them separately wastes vocabulary slots.
2. **Punctuation removal** — Only tokens passing `word.isalpha()` are kept. Tokens like `,`, `.`, `--`, `''` are removed.
3. **Vocabulary cutoff (top 10,000)** — Rare words mapped to `<UNK>`. Keeps output layer tractable.

In [ ]:
print("Sample BEFORE preprocessing:")
print(all_raw_words[:20])
print()

corpus = [w.lower() for w in all_raw_words if w.isalpha()]

print("Sample AFTER preprocessing:")
print(corpus[:20])
print()
print(f"Raw tokens     : {len(all_raw_words):,}")
print(f"Cleaned tokens : {len(corpus):,}")
print(f"Removed        : {len(all_raw_words)-len(corpus):,}  ({1-len(corpus)/len(all_raw_words):.1%})")

## 2.2 — Vocabulary Construction and Coverage Analysis

In [ ]:
VOCAB_SIZE = 10_000

freq       = Counter(corpus)
top_words  = [w for w, _ in freq.most_common(VOCAB_SIZE - 1)]
vocabulary = ["<UNK>"] + top_words
word_to_id = {w: i for i, w in enumerate(vocabulary)}
id_to_word = {i: w for w, i in word_to_id.items()}
token_ids  = [word_to_id.get(w, 0) for w in corpus]

oov_count  = sum(1 for w in corpus if w not in word_to_id)
coverage   = 1 - oov_count / len(corpus)

print(f"Vocabulary size   : {len(vocabulary):,}")
print(f"Corpus coverage   : {coverage:.2%} of tokens are in-vocabulary")
print(f"OOV rate          : {oov_count/len(corpus):.2%}")
print(f"Encoded tokens    : {len(token_ids):,}")
print()
print("Most common words (top 10):", [w for w,_ in freq.most_common(10)])

In [ ]:
sorted_freq  = [c for _, c in freq.most_common()]
total_tokens = len(corpus)
cumulative, cum_list = 0, []
for c in sorted_freq:
    cumulative += c
    cum_list.append(cumulative / total_tokens)

vocab_sizes = list(range(500, 25001, 500))
coverages   = [cum_list[min(vs-1, len(cum_list)-1)] for vs in vocab_sizes]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(vocab_sizes, [c*100 for c in coverages], color="steelblue", linewidth=2)
axes[0].axvline(VOCAB_SIZE, color="red", linestyle="--", linewidth=1.5, label=f"Selected: {VOCAB_SIZE:,} words")
axes[0].axhline(coverage*100, color="green", linestyle=":", linewidth=1.5, label=f"Coverage: {coverage:.1%}")
axes[0].set_xlabel("Vocabulary Size")
axes[0].set_ylabel("Corpus Coverage (%)")
axes[0].set_title("Coverage vs Vocabulary Size", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

top30_clean = freq.most_common(30)
axes[1].barh([w for w,_ in top30_clean][::-1], [c for _,c in top30_clean][::-1], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Frequency")
axes[1].set_title("Top 30 Words After Preprocessing", fontweight="bold")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("vis3_vocabulary.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.3 — Sequence Construction

A **sliding window** of length 20 extracts input/output pairs: input = 20 word IDs, target = next word.
Stride=3 reduces total sequence count for memory efficiency while covering the full corpus.

In [ ]:
CONTEXT_LEN = 20
STRIDE      = 3

inputs, targets = [], []
for start in range(0, len(token_ids) - CONTEXT_LEN, STRIDE):
    inputs.append(token_ids[start : start + CONTEXT_LEN])
    targets.append(token_ids[start + CONTEXT_LEN])

inputs  = np.array(inputs,  dtype=np.int32)
targets = np.array(targets, dtype=np.int32)

print(f"Total sequences : {len(inputs):,}")
print(f"Input shape     : {inputs.shape}")
print(f"Target shape    : {targets.shape}")
print()
print("Example input  :", [id_to_word[i] for i in inputs[100]])
print("Example target :", repr(id_to_word[targets[100]]))

## 2.4 — Train / Validation / Test Split

Split is **chronological** (not random) to preserve natural text order.
Random shuffling would leak adjacent-sentence context across splits.

In [ ]:
total     = len(inputs)
train_cut = int(0.80 * total)
val_cut   = int(0.90 * total)

X_train, y_train = inputs[:train_cut],        targets[:train_cut]
X_val,   y_val   = inputs[train_cut:val_cut], targets[train_cut:val_cut]
X_test,  y_test  = inputs[val_cut:],          targets[val_cut:]

print(f"Train      : {len(X_train):>8,} sequences  ({len(X_train)/total:.0%})")
print(f"Validation : {len(X_val):>8,} sequences  ({len(X_val)/total:.0%})")
print(f"Test       : {len(X_test):>8,} sequences  ({len(X_test)/total:.0%})")

fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(["Dataset"], [len(X_train)], color="steelblue",  label=f"Train ({len(X_train):,})")
ax.barh(["Dataset"], [len(X_val)],   color="darkorange", label=f"Val ({len(X_val):,})",   left=[len(X_train)])
ax.barh(["Dataset"], [len(X_test)],  color="green",      label=f"Test ({len(X_test):,})", left=[len(X_train)+len(X_val)])
ax.set_xlabel("Number of Sequences")
ax.set_title("Train / Validation / Test Split (80 / 10 / 10)", fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("vis4_split.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Part 3: Model Architectures

All models share: `Input(20) -> Embedding(10000,64) -> [arch] -> Dropout(0.3) -> Dense(10000, softmax)`

The **Functional API** is used — it builds the model graph immediately so `.summary()` always
shows correct parameter counts. Optimizer=Adam, Loss=Sparse Categorical Cross-Entropy.

## 3.1 — Model 1: Dense Neural Network

Flattens the 20x64 embedding to a 1,280-dim vector, then Dense(256, ReLU) → Dense(128, ReLU).
**ReLU** avoids vanishing gradients. No sequential inductive bias — treats all positions equally.

**Why include it:** Serves as a fast non-sequential baseline to show the value of RNNs.

**Hyperparameters:** Hidden dims [256, 128], Dropout=0.3, Adam, batch_size=512

In [ ]:
EMBED_DIM = 64

inp1 = keras.Input(shape=(CONTEXT_LEN,), name="word_ids")
x1   = keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding")(inp1)
x1   = keras.layers.Flatten(name="flatten")(x1)
x1   = keras.layers.Dense(256, activation="relu", name="dense_1")(x1)
x1   = keras.layers.Dropout(0.3, name="dropout_1")(x1)
x1   = keras.layers.Dense(128, activation="relu", name="dense_2")(x1)
x1   = keras.layers.Dropout(0.3, name="dropout_2")(x1)
out1 = keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(x1)

dense_model = keras.Model(inputs=inp1, outputs=out1, name="Dense_NN")
dense_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
dense_model.summary()

## 3.2 — Model 2: LSTM

Processes the token sequence step-by-step with three gates (forget, input, output).
Gates control information flow through the cell state and hidden state,
allowing the LSTM to capture long-range dependencies a Dense NN cannot.

**Why include it:** Classical RNN for language modeling; gating solves the vanishing gradient problem.

**Hyperparameters:** Units=128, Dropout=0.3, Adam, batch_size=256

In [ ]:
inp2 = keras.Input(shape=(CONTEXT_LEN,), name="word_ids")
x2   = keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding")(inp2)
x2   = keras.layers.LSTM(128, name="lstm")(x2)
x2   = keras.layers.Dropout(0.3, name="dropout")(x2)
out2 = keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(x2)

lstm_model = keras.Model(inputs=inp2, outputs=out2, name="LSTM")
lstm_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
lstm_model.summary()

## 3.3 — Model 3: GRU

Simplifies the LSTM: merges cell+hidden state into one, uses only two gates (reset, update).
~25% fewer parameters than LSTM; trains faster with comparable perplexity.

**Why include it:** Tests whether LSTM complexity provides measurable benefit over a simpler GRU.

**Hyperparameters:** Units=128, Dropout=0.3, Adam, batch_size=256

In [ ]:
inp3 = keras.Input(shape=(CONTEXT_LEN,), name="word_ids")
x3   = keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding")(inp3)
x3   = keras.layers.GRU(128, name="gru")(x3)
x3   = keras.layers.Dropout(0.3, name="dropout")(x3)
out3 = keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(x3)

gru_model = keras.Model(inputs=inp3, outputs=out3, name="GRU")
gru_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
gru_model.summary()

## 3.4 — Model 4: Bidirectional LSTM

A **Bidirectional LSTM** runs two LSTM sub-layers simultaneously: one reads the token
sequence left-to-right, the other right-to-left. Their hidden states are concatenated,
giving the model access to both past and future context at each position.
We use 64 units per direction (128-dim concatenated output) so the total parameter
count stays comparable to the unidirectional models.
Trade-off: roughly 2x the per-step compute of a plain LSTM, but often better
at capturing phrase-level patterns.

In [ ]:
inp4 = keras.Input(shape=(CONTEXT_LEN,), name="word_ids")
x4   = keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="embedding")(inp4)
x4   = keras.layers.Bidirectional(keras.layers.LSTM(64), name="bilstm")(x4)
x4   = keras.layers.Dropout(0.3, name="dropout")(x4)
out4 = keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(x4)
bilstm_model = keras.Model(inputs=inp4, outputs=out4, name="BiLSTM")
bilstm_model.compile(optimizer="adam",
                     loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])
bilstm_model.summary()

In [ ]:
model_names  = ["Dense NN", "LSTM", "GRU", "BiLSTM"]
param_counts = [
    dense_model.count_params(), lstm_model.count_params(),
    gru_model.count_params(),   bilstm_model.count_params()
]

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ["steelblue", "darkorange", "green", "purple"]
bars    = ax.bar(model_names, param_counts, color=colors, edgecolor="black", width=0.5)
for bar, p in zip(bars, param_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10_000,
            f"{p:,}", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_ylabel("Total Parameters", fontsize=12)
ax.set_title("Model Size Comparison — Trainable Parameters",
             fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

---
# Part 4: Training

All four models trained with identical settings for fair comparison:
- **Early stopping** (patience=2): stops when val_loss stops improving, restores best weights
- **Dense NN batch=512**: no BPTT overhead, large batches are efficient
- **LSTM/GRU batch=256**: recurrent backprop is memory-intensive
- **Max 10 epochs**: early stopping typically fires at epoch 3-6

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True, verbose=1
)

print("=" * 50)
print("  Training Model 1: Dense NN")
print("=" * 50)
dense_history = dense_model.fit(
    X_train, y_train, epochs=10, batch_size=512,
    validation_data=(X_val, y_val), callbacks=[early_stop]
)
print("Dense NN done.")

In [ ]:
print("=" * 50)
print("  Training Model 2: LSTM")
print("=" * 50)
lstm_history = lstm_model.fit(
    X_train, y_train, epochs=10, batch_size=256,
    validation_data=(X_val, y_val), callbacks=[early_stop]
)
print("LSTM done.")

In [ ]:
print("=" * 50)
print("  Training Model 3: GRU")
print("=" * 50)
gru_history = gru_model.fit(
    X_train, y_train, epochs=10, batch_size=256,
    validation_data=(X_val, y_val), callbacks=[early_stop]
)
print("GRU done.")

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
print("=" * 50)
print("  Training Model 4: BiLSTM")
print("=" * 50)
bilstm_history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)
print("BiLSTM done.")

In [ ]:
histories = {
    "Dense NN": dense_history,
    "LSTM":     lstm_history,
    "GRU":      gru_history,
    "BiLSTM":   bilstm_history,
}
clrs = {"Dense NN": "steelblue", "LSTM": "darkorange",
        "GRU": "green", "BiLSTM": "purple"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Training Curves — Dense NN vs LSTM vs GRU vs BiLSTM",
             fontsize=15, fontweight="bold")

for name, hist in histories.items():
    c = clrs[name]
    axes[0, 0].plot(hist.history["val_accuracy"], marker="o", label=name, color=c)
    axes[0, 1].plot(hist.history["val_loss"],     marker="o", label=name, color=c)
    axes[1, 0].plot(hist.history["val_accuracy"], "-",  color=c, label=f"{name} val")
    axes[1, 0].plot(hist.history["accuracy"],     "--", color=c, alpha=0.5)
    axes[1, 1].plot(hist.history["val_loss"],     "-",  color=c, label=f"{name} val")
    axes[1, 1].plot(hist.history["loss"],         "--", color=c, alpha=0.5)

for ax in axes.flat:
    ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3)
axes[0,0].set_title("Validation Accuracy — All Models"); axes[0,0].set_ylabel("Accuracy"); axes[0,0].legend()
axes[0,1].set_title("Validation Loss — All Models");     axes[0,1].set_ylabel("Loss");     axes[0,1].legend()
axes[1,0].set_title("Train (dashed) vs Val (solid) Accuracy"); axes[1,0].set_ylabel("Accuracy"); axes[1,0].legend()
axes[1,1].set_title("Train (dashed) vs Val (solid) Loss");     axes[1,1].set_ylabel("Loss");     axes[1,1].legend()
plt.tight_layout()
plt.show()

## 4.2 — Perplexity Learning Curves

Perplexity = exp(cross-entropy loss) is the primary language model metric.
Plotting it directly during training shows how quickly each model converges
and how large the train/validation gap is at each epoch.

In [ ]:
import math

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Perplexity Learning Curves — All 4 Models",
             fontsize=14, fontweight="bold")

for name, hist in histories.items():
    c = clrs[name]
    val_ppl   = [math.exp(l) for l in hist.history["val_loss"]]
    train_ppl = [math.exp(l) for l in hist.history["loss"]]
    epochs    = range(1, len(val_ppl) + 1)
    axes[0].plot(epochs, val_ppl,   marker="o", color=c, label=name)
    axes[1].plot(epochs, val_ppl,   linestyle="-",  color=c, label=f"{name} val")
    axes[1].plot(epochs, train_ppl, linestyle="--", color=c, alpha=0.45)

axes[0].set_title("Validation Perplexity per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Perplexity (lower = better)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Train (dashed) vs Validation (solid) Perplexity")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Perplexity")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 5: Validation Results

**Perplexity = exp(cross-entropy loss)**. Perplexity of N means the model is as uncertain as
choosing uniformly among N words. A random baseline over 10,000 words gives perplexity ~10,000.
Lower perplexity = better model. Test set is reserved for Part 6.

In [ ]:
import math
val_results = {}
all_names  = ["Dense NN",    "LSTM",        "GRU",        "BiLSTM"]
all_hists  = [dense_history, lstm_history,  gru_history,  bilstm_history]
all_models = [dense_model,   lstm_model,    gru_model,    bilstm_model]

print("=" * 60)
print(f"  {'Model':12s}  {'Val Loss':>10s}  {'Perplexity':>12s}  {'Accuracy':>10s}")
print("=" * 60)
for name, hist, model in zip(all_names, all_hists, all_models):
    best_ep   = int(np.argmin(hist.history["val_loss"]))
    best_loss = hist.history["val_loss"][best_ep]
    best_acc  = hist.history["val_accuracy"][best_ep]
    ppl       = math.exp(best_loss)
    val_results[name] = {"loss": best_loss, "ppl": ppl,
                         "acc":  best_acc,  "model": model}
    print(f"  {name:12s}  {best_loss:>10.4f}  {ppl:>12.2f}  {best_acc:>10.4f}")
print("=" * 60)
best_name = min(val_results, key=lambda n: val_results[n]["ppl"])
print(f"Best model: {best_name}  (lowest val perplexity)")

In [ ]:
names_v  = list(val_results.keys())
ppls_v   = [val_results[n]["ppl"] for n in names_v]
accs_v   = [val_results[n]["acc"] * 100 for n in names_v]
colors_v = ["steelblue", "darkorange", "green", "purple"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Validation Results — All 4 Models Compared",
             fontsize=14, fontweight="bold")

b1 = axes[0].bar(names_v, ppls_v, color=colors_v, edgecolor="black")
for bar, v in zip(b1, ppls_v):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+3, f"{v:.1f}",
                ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Validation Perplexity by Model")
axes[0].set_ylabel("Perplexity (lower = better)")

b2 = axes[1].bar(names_v, accs_v, color=colors_v, edgecolor="black")
for bar, v in zip(b2, accs_v):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.05, f"{v:.2f}%",
                ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Validation Accuracy by Model")
axes[1].set_ylabel("Accuracy (%)")

plt.tight_layout()
plt.show()

## 5.2 — Comparison Against Random Baseline

A naive **random baseline** assigns uniform probability 1/V to every word, giving
perplexity = V = 10,000. Any model with learned structure must beat this.
The chart below makes the improvement concrete.

In [ ]:
random_ppl = VOCAB_SIZE
print(f"Random baseline perplexity : {random_ppl:,}")
best_ppl = val_results[best_name]['ppl']
print(f"Best model ({best_name}) ppl: {best_ppl:.2f}")
print(f"Improvement over random    : {random_ppl/best_ppl:.1f}x")

fig, ax = plt.subplots(figsize=(9, 5))
names_b  = list(val_results.keys())
ppls_b   = [val_results[n]['ppl'] for n in names_b]
clrs_b   = ['steelblue', 'darkorange', 'green', 'purple']
bars_b   = ax.bar(names_b, ppls_b, color=clrs_b,
                  edgecolor='black', width=0.5, zorder=3)
ax.axhline(random_ppl, color='red', linestyle='--', linewidth=2,
           label=f'Random baseline ({random_ppl:,})', zorder=2)
for bar, p in zip(bars_b, ppls_b):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            f'{p:.1f}', ha='center', va='bottom',
            fontweight='bold', fontsize=11)
ax.set_ylabel('Perplexity (lower = better)', fontsize=12)
ax.set_title('Validation Perplexity vs Random Baseline',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(ppls_b) * 1.25)
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

## 5.3 — Unigram Language Model Baseline

The **unigram baseline** is more informative than random: it predicts each word
with probability proportional to its frequency in the training corpus — no context used.
Any model using context must beat this to prove it learned sequential structure,
not just word frequencies.

In [ ]:
import math
from collections import Counter

# Build unigram distribution from training targets
unigram_counts = Counter(int(x) for x in y_train)
total_train    = len(y_train)
unigram_probs  = {wid: cnt / total_train
                  for wid, cnt in unigram_counts.items()}

# Evaluate on validation targets
log_sum = sum(
    math.log(unigram_probs.get(int(t), 1.0 / VOCAB_SIZE))
    for t in y_val
)
unigram_ppl = math.exp(-log_sum / len(y_val))

random_ppl  = VOCAB_SIZE
best_ppl    = val_results[best_name]['ppl']

print(f"Random  baseline perplexity : {random_ppl:>10,}")
print(f"Unigram baseline perplexity : {unigram_ppl:>10.2f}")
print(f"Best model ({best_name}) ppl: {best_ppl:>10.2f}")
print(f"")
print(f"Improvement over random     : {random_ppl  / best_ppl:.1f}x")
print(f"Improvement over unigram    : {unigram_ppl / best_ppl:.2f}x")

# Chart: three baselines side by side
fig, ax = plt.subplots(figsize=(10, 5))
labels  = ["Random\nbaseline", "Unigram\nbaseline",
            "Dense NN", "LSTM", "GRU", "BiLSTM"]
ppls    = [random_ppl, unigram_ppl] + [val_results[n]['ppl']
            for n in val_results]
colors  = ["#ef9a9a", "#ffcc80",
            "steelblue", "darkorange", "green", "purple"]
bars    = ax.bar(labels, ppls, color=colors, edgecolor="black", width=0.6)
for bar, p in zip(bars, ppls):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(ppls)*0.01,
            f"{p:,.1f}", ha="center", va="bottom",
            fontweight="bold", fontsize=10)
ax.set_ylabel("Perplexity (lower = better)", fontsize=12)
ax.set_title("Model Perplexity vs Baselines",
             fontsize=14, fontweight="bold")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 6: Analysis and Final Evaluation

## 6.1 — Final Test Evaluation

Test set evaluated **only once**, on the best model selected by validation perplexity.
Using the test set to guide model selection would inflate reported performance (data leakage).

In [ ]:
best_model          = val_results[best_name]["model"]
test_loss, test_acc = best_model.evaluate(X_test, y_test, batch_size=512, verbose=0)
test_ppl            = float(np.exp(test_loss))

val_ppl_best = val_results[best_name]["ppl"]
val_acc_best = val_results[best_name]["acc"]

print(f"Best model : {best_name}  (val perplexity = {val_ppl_best:.2f})")
print()
print("=" * 45)
print(f"  FINAL TEST RESULTS  —  {best_name}")
print("=" * 45)
print(f"  Test Loss        : {test_loss:.4f}")
print(f"  Test Perplexity  : {test_ppl:.2f}")
print(f"  Test Accuracy    : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print("=" * 45)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric_vals, ylabel, title in [
    (axes[0], [val_ppl_best, test_ppl],        "Perplexity",  "Perplexity"),
    (axes[1], [val_acc_best*100, test_acc*100], "Accuracy (%)", "Accuracy")
]:
    brs = ax.bar(["Validation", "Test"], metric_vals, color=["steelblue","darkorange"], edgecolor="white", width=0.4)
    for bar, v in zip(brs, metric_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f"{v:.2f}", ha="center", fontweight="bold", fontsize=13)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{best_name} — {title}", fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle(f"Final Evaluation — {best_name}: Validation vs Test", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("vis8_final_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.2 — Top-K Next-Word Predictions

For each of three thematically distinct context phrases, we display the
**top-10 words** the best model assigns highest probability to as the next token.
Thematically coherent predictions (e.g. *whale* or *sea* after a Moby Dick context)
show the model learned genuine semantic associations, not just word frequency.

In [ ]:
def top_k_preds(model, context_words, k=10):
    unk_id = word_to_id.get('<UNK>', 1)
    ctx    = [word_to_id.get(w, unk_id) for w in context_words]
    ctx    = (ctx + [0] * CONTEXT_LEN)[:CONTEXT_LEN]
    probs  = model.predict(np.array([ctx]), verbose=0)[0]
    top_i  = np.argsort(probs)[::-1][:k]
    return [id_to_word.get(int(j), '<UNK>') for j in top_i], probs[top_i]

demo_contexts = [
    ["the", "great", "white", "whale", "was", "seen"],
    ["she", "had", "never", "seen", "such", "a"],
    ["and", "god", "said", "let", "there", "be"],
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Top-10 Predicted Next Words — {best_name} Model",
             fontsize=14, fontweight="bold")

for ax, ctx in zip(axes, demo_contexts):
    words, probs = top_k_preds(best_model, ctx)
    bar_c = ["#9E9E9E" if w == "<UNK>" else "#2196F3" for w in words]
    ax.barh(range(len(words)-1, -1, -1), probs,
            color=bar_c, edgecolor="black")
    ax.set_yticks(range(len(words)-1, -1, -1))
    ax.set_yticklabels(words, fontsize=11)
    ax.set_xlabel("Probability")
    ax.set_title('Context: "...' + ' '.join(ctx[-4:]) + '"', fontsize=10)
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 6.3 — Word Embedding Visualization (PCA)

The shared `Embedding` layer learned a 64-dimensional vector for each vocabulary word.
We apply **PCA** to project these into 2D and check whether semantically related words
cluster together — the classic hallmark of a well-trained embedding space.
Gray dots are the full 10,000-word vocabulary; colored dots are handpicked thematic groups.

In [ ]:
from sklearn.decomposition import PCA

emb_w  = best_model.get_layer("embedding").get_weights()[0]  # (10000, 64)
pca2   = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(emb_w)

groups = {
    "Maritime":  (["sea",  "ocean",  "ship",   "whale",  "sailor",  "captain"],  "#2196F3"),
    "Religion":  (["god",  "lord",   "heaven", "prayer", "holy",    "church"],   "#9C27B0"),
    "Emotion":   (["love", "heart",  "soul",   "joy",    "fear",    "hope"],     "#4CAF50"),
    "Conflict":  (["war",  "battle", "sword",  "enemy",  "fight",   "army"],     "#F44336"),
    "Royalty":   (["king", "queen",  "prince", "crown",  "throne",  "court"],    "#FF9800"),
}

fig, ax = plt.subplots(figsize=(13, 10))
ax.scatter(coords[:, 0], coords[:, 1], alpha=0.04, s=4, color="lightgray")

from matplotlib.lines import Line2D
legend_handles = []
for gname, (words, color) in groups.items():
    for w in words:
        if w in word_to_id:
            x, y = coords[word_to_id[w]]
            ax.scatter(x, y, color=color, s=150, zorder=5,
                       edgecolors="black", linewidths=0.5)
            ax.annotate(w, (x, y), fontsize=9, fontweight="bold",
                        xytext=(5, 4), textcoords="offset points", color=color)
    legend_handles.append(
        Line2D([0],[0], marker="o", color="w", markerfacecolor=color,
               markeredgecolor="black", markersize=11, label=gname)
    )

ax.set_title(
    f"Word Embedding Space — PCA Projection ({best_name})\n"
    f"PC1: {pca2.explained_variance_ratio_[0]*100:.1f}%  ·  "
    f"PC2: {pca2.explained_variance_ratio_[1]*100:.1f}%  variance explained",
    fontsize=13
)
ax.set_xlabel("Principal Component 1", fontsize=11)
ax.set_ylabel("Principal Component 2", fontsize=11)
ax.legend(handles=legend_handles, loc="upper right", fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
# Part 7: Text Generation Demo

The best model generates text autoregressively — each predicted word is appended to the input
and the window slides forward. This qualitatively verifies the model learned language patterns.

**Temperature** controls randomness: 0.5 = conservative, 0.8 = balanced, 1.2 = creative.

In [ ]:
def generate_text(model, seed_words, num_words=40, temperature=0.8):
    gen_seq = [word_to_id.get(w, 0) for w in seed_words]
    output  = list(seed_words)
    for _ in range(num_words):
        x     = keras.utils.pad_sequences([gen_seq[-CONTEXT_LEN:]], maxlen=CONTEXT_LEN)
        probs = model.predict(x, verbose=0)[0].astype("float64")
        probs = probs ** (1.0 / temperature)
        probs /= probs.sum()
        next_id = np.random.choice(len(probs), p=probs)
        output.append(id_to_word[next_id])
        gen_seq.append(next_id)
    return " ".join(output)

seeds = [
    ["the", "captain", "looked", "out", "upon", "the"],
    ["she", "had", "never", "seen", "such", "a"],
    ["it", "was", "a", "dark", "and", "stormy"]
]
print(f"=== Text Generation with {best_name} (temperature=0.8) ===\n")
for seed in seeds:
    print(f"Seed: {seed}")
    print(generate_text(best_model, seed, num_words=40, temperature=0.8))
    print()

In [ ]:
np.random.seed(42)
seed_demo = ["the", "old", "man", "walked", "slowly", "toward"]
temps     = [0.5, 0.8, 1.2]
labels    = ["Temperature = 0.5  (conservative)", "Temperature = 0.8  (balanced)", "Temperature = 1.2  (creative)"]

fig, axes = plt.subplots(3, 1, figsize=(14, 7))
for i, (temp, label) in enumerate(zip(temps, labels)):
    text = generate_text(best_model, seed_demo, num_words=30, temperature=temp)
    axes[i].text(0.01, 0.5, text, transform=axes[i].transAxes, fontsize=10, va="center",
                 bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
    axes[i].set_title(label, fontweight="bold", loc="left", fontsize=11)
    axes[i].set_xticks([]); axes[i].set_yticks([])

plt.suptitle(f"Text Generation at Different Temperatures — {best_name}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("vis9_temperature.png", dpi=150, bbox_inches="tight")
plt.show()

### Temperature Analysis

- **Temperature = 0.5 (conservative):** Low temperature *sharpens* the softmax distribution,
  concentrating nearly all probability mass on the single most-likely token. In this corpus
  `<UNK>` is structurally the most probable token at many positions (it represents ~4% of
  all vocabulary slots). Once selected, `<UNK>` enters the context window and biases the
  next prediction toward `<UNK>` again, creating a cascade. This is a well-known artifact
  of word-level models with an explicit `<UNK>` token; character-level or subword (BPE)
  models avoid it.

- **Temperature = 0.8 (balanced):** Produces the most coherent sequences. Vocabulary from
  multiple books appears naturally (Austen sentence structure, Moby Dick maritime terms).

- **Temperature = 1.2 (creative):** Higher entropy allows rarer words to surface. Biblical
  vocabulary (ephah, tyre, david) appears — correctly sampled from the King James Bible
  portion of the corpus. Grammar loosens but diversity increases.

---
# Part 8: Project Report — Language Modeling on Project Gutenberg

## 1. Introduction

For this project, I built and compared four different neural network models to do word-level language modeling using a collection of classic English books from Project Gutenberg. The basic idea behind a language model is straightforward: given a sequence of words, predict what word comes next. This is one of the oldest problems in natural language processing and it underlies modern tools like autocomplete, machine translation, and text generation.

I used four architectures: a Dense Neural Network (no memory of word order), an LSTM, a GRU, and a Bidirectional LSTM. The main evaluation metric is **perplexity** — lower is better. I also compared all four models against two baselines: a random baseline and a unigram baseline, to put the numbers in proper context.

---

## 2. Dataset

The data comes from the **NLTK Gutenberg corpus**, a collection of 18 public domain books that download automatically without any manual setup. The books cover a wide mix of writing styles and time periods:

- **Novels:** Moby Dick, Emma, Sense and Sensibility, Persuasion
- **Shakespeare plays:** Hamlet, Macbeth, Julius Caesar
- **Religious text:** King James Bible — the single largest book at 1,010,654 tokens (~38% of the corpus)
- **Poetry:** Walt Whitman, William Blake
- **Other prose:** G.K. Chesterton, Lewis Carroll, Maria Edgeworth

**Raw corpus stats:**
- Total raw tokens: 2,621,613
- Unique tokens: 51,156
- Number of books: 18

Word frequency in this corpus follows **Zipf's Law** — a small number of words appear very often while most words are rare. The top 10 words are: *the, and, of, to, a, in, i, that, he, it* — all common English function words. The log-log frequency plot in Part 1 shows a straight line, which confirms this.

---

## 3. Preprocessing

I applied three preprocessing steps to clean the raw text before building any models.

**Step 1 — Lowercase everything.**
Words like "The" and "the" are the same word. Converting to lowercase prevents the model from treating them as different vocabulary entries.

**Step 2 — Remove punctuation and non-alphabetic tokens.**
Only tokens that pass `word.isalpha()` were kept. This removed brackets, numbers, commas, quotes, and dashes. In total, **486,213 tokens were removed** (18.5% of the raw corpus), leaving **2,135,400 clean tokens**.

**Step 3 — Cut vocabulary to 10,000 words.**
The corpus has over 51,000 unique words, but most appear only once or twice. I kept the 10,000 most frequent words and replaced everything else with `<UNK>`. These 10,000 words cover **96.21% of the corpus** (OOV rate = 3.79%), so very little information is lost. The vocabulary coverage curve in Part 2 shows that going beyond 10,000 gives very little additional coverage.

**Sequence construction:**
I used a sliding window of length 20 with a stride of 3 to build input-output pairs. Each input is 20 consecutive word IDs and the target is the 21st word. Stride 3 reduces redundancy without skipping much of the corpus. This produced **711,794 sequences** total.

**Train / Validation / Test Split:**

| Split | Sequences | Percentage |
|---|---|---|
| Train | 569,435 | 80% |
| Validation | 71,179 | 10% |
| Test | 71,180 | 10% |

The split is **chronological**, not random. This is important — a random split would let the model see words from the middle of a sequence during training and predict the beginning, which would be unrealistic and would inflate performance numbers.

---

## 4. Model Architectures

All four models solve the same problem: given 20 word IDs as input, predict which of the 10,000 vocabulary words comes next. Every model shares three components:

- **Embedding layer (10,000 × 64):** Maps each word ID to a 64-dimensional learnable vector. This layer learns to put similar words close together in vector space.
- **Dropout (rate = 0.3):** Randomly sets 30% of activations to zero during training. This prevents the model from memorizing training data too closely.
- **Output layer (Dense, 10,000 units, softmax):** Produces a probability for every word in the vocabulary.

### Model 1: Dense Neural Network — 2,290,832 parameters
The 20 × 64 embedding output is flattened into a single 1,280-dimensional vector, then passed through Dense(256, ReLU) → Dropout → Dense(128, ReLU) → Dropout → output. This model has no concept of word order — it treats the 20 context words as a flat set of features.

### Model 2: LSTM — 2,028,816 parameters
An LSTM reads the word sequence one step at a time. Three internal gates (forget, input, output) decide what to remember and what to discard at each step. This allows the model to capture dependencies between words that are far apart. 128 hidden units.

### Model 3: GRU — 2,004,496 parameters
The GRU is a simplified LSTM that uses only two gates (reset and update) and combines the memory cell and hidden state into one. It has roughly 25% fewer recurrent parameters than the LSTM but performs at a similar level. 128 units.

### Model 4: Bidirectional LSTM — 1,996,048 parameters
Two LSTM layers run at the same time: one reads left to right, one reads right to left. Their outputs are concatenated into a 128-dimensional vector (64 per direction). The idea is that context from both directions might help. Despite having the most complex design, it has the fewest total parameters because each direction only uses 64 units instead of 128.

The parameter comparison chart in Part 3 shows all four model sizes side by side.

---

## 5. Training Setup

All models were trained with the same configuration for a fair comparison:

| Setting | Value |
|---|---|
| Optimizer | Adam |
| Loss | Sparse categorical cross-entropy |
| Max epochs | 10 |
| Early stopping patience | 2 epochs |
| Restore best weights | Yes |
| Batch size (Dense NN) | 512 |
| Batch size (LSTM / GRU / BiLSTM) | 256 |
| Dropout rate | 0.3 |

Early stopping monitors validation loss and stops training if it does not improve for 2 consecutive epochs. A fresh callback was created for each model to avoid any shared state between training runs.

---

## 6. Results

### What is Perplexity?

Perplexity = exp(cross-entropy loss). A perplexity of N means the model is roughly as uncertain as if it had to pick from N equally likely choices at each step. Lower is better. A model that just guesses randomly over 10,000 words has perplexity = 10,000. A model that only looks at word frequency (no context) gives perplexity = 852.64 on this corpus.

### Validation Results — All 4 Models

| Model | Parameters | Val Perplexity | Val Accuracy |
|---|---|---|---|
| Dense NN | 2,290,832 | 592.5 | 11.41% |
| LSTM | 2,028,816 | 553.4 | 12.05% |
| **GRU** | **2,004,496** | **503.9** | **12.30%** |
| BiLSTM | 1,996,048 | 534.0 | 12.30% |

See the validation bar charts in Part 5 and training curves in Part 4.

### Comparison Against Baselines

| Model / Baseline | Perplexity | vs. GRU |
|---|---|---|
| Random (uniform guess) | 10,000 | 19.8x worse |
| Unigram (word frequency only) | 852.64 | 1.69x worse |
| Dense NN | 592.5 | 1.18x worse |
| LSTM | 553.4 | 1.10x worse |
| BiLSTM | 534.0 | 1.06x worse |
| **GRU (best)** | **503.9** | — |

The GRU beats the unigram baseline by **1.69x** and beats random by **19.8x**. See the baseline comparison chart in Part 5.

### Final Test Evaluation — GRU Only

The test set was used exactly once, only on the winning model (GRU):

| | Perplexity | Accuracy |
|---|---|---|
| Validation | 503.9 | 12.30% |
| **Test** | **751.41** | **12.01%** |

The test perplexity (751) is higher than the validation perplexity (504). This is expected and not a sign of a bug or error. Because the split is chronological, the test set is the very last portion of the corpus — the text furthest from what the model trained on. There is no data leakage; this reflects real-world behavior on genuinely new text.

---

## 7. Analysis and Discussion

### Why did GRU win?

The GRU reached the lowest validation perplexity (503.9) with the second-fewest parameters and the fastest training time among the recurrent models. Looking at the perplexity learning curves (Part 4), the GRU drops quickly and levels off around epoch 3-4, while the LSTM needs more epochs to converge and the Dense NN's validation loss starts going up after epoch 2.

The Dense NN's poor performance makes sense: it has no concept of word order. When you flatten the 20 embeddings into one vector, you lose all sequential information. Language is inherently sequential — "the dog bit the man" and "the man bit the dog" become the same input to a Dense NN, but they have very different meanings.

### Why did BiLSTM underperform GRU?

The BiLSTM was expected to do well because it can see context from both directions. But for language modeling, the task is specifically to predict the next word using only past words. During training, the backward LSTM looks at future words to help predict current words, which adds noise rather than signal for this particular task. More complex is not always better — task alignment matters. The BiLSTM would likely outperform the GRU on tasks like text classification or named entity recognition, where full-sentence context helps.

### Overfitting

The train vs. validation perplexity gap (right side of Part 4 chart) is large: training perplexity drops to around 100-150 while validation stays around 500. This means the models memorize the training sequences better than they generalize to new text. This is common with a relatively small corpus (only 18 books). Dropout and early stopping reduced this gap but did not eliminate it. Using a larger dataset or more regularization would help.

### What the model actually learned

The word embedding PCA visualization (Part 6) shows that the GRU learned meaningful structure just from predicting words. "Captain" sits isolated from the main cluster — it appears almost exclusively in Moby Dick contexts, so it has a unique embedding. "Holy" and "god" are pulled in different directions even though both are religious words — they appear in very different grammatical positions in the Bible. Maritime words (whale, sea, sailor) form a loose cluster. This is good evidence the model learned something about word meaning, not just frequency.

The top-K prediction charts (Part 6) show that the model's predictions shift depending on context. After a Moby Dick seed phrase, maritime words get higher probability. After a biblical seed phrase, religious vocabulary rises. This context-sensitivity proves the model is doing sequential reasoning, not just counting words.

### Text Generation

At temperature 0.8, the model generates text like: *"forth syme look in the south house and he he deny the living and he will not consume them therefore make him according to..."* — mixing a character name from Chesterton ("syme") with King James Bible phrasing. At temperature 1.2, biblical place names (Gibeon) and objects (barley) appear. This reflects the composition of the training corpus accurately.

At temperature 0.5, the model collapses into repeating `<UNK>` tokens. This is a known artifact: at low temperature, the model concentrates almost all probability on the single most likely token. Since `<UNK>` appears very frequently in literary text (representing all rare words), it wins and then biases the next prediction toward `<UNK>` again. Subword tokenization (BPE) would eliminate this problem.

---

## 8. Conclusion

The GRU is the best architecture for word-level language modeling on the Project Gutenberg corpus. It achieves perplexity 503.9 on validation — **19.8x better than random** and **1.69x better than a frequency-only unigram model**. The sequential processing of recurrent models is clearly important for this task, as shown by the Dense NN's worse performance despite having the most parameters.

An honest observation: the 1.69x improvement over the unigram baseline is modest. Most of the predictability in this corpus comes from word frequency patterns rather than sequential context. This is a known characteristic of small language models trained at this scale. Modern large language models (GPT-4, Claude) show dramatic improvements over unigram baselines because they use much deeper networks, longer context windows, billions of training tokens, and subword tokenization.

**Possible next steps:**
- Stacked LSTM or GRU layers (deeper = more capacity)
- Pre-trained word embeddings like GloVe or Word2Vec
- Subword tokenization (BPE) to eliminate the `<UNK>` problem
- Attention mechanisms and transformer-based architectures
- Training on a larger and more consistent corpus (e.g., all of Wikipedia)
